<a href="https://colab.research.google.com/github/MatteoBaraldi/Machine-Learning-for-Bioengineering/blob/giovanni/MOD-1/notebooks/08_age_prediction_eda_classification_cv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Age Prediction: data preparation, Exploratory Data Analysis (EDA), and regression with cross-validation (CV) & nested CV

In [ ]:
#preparation samebut before all the computation we need the threshold

## Preparing data for a classification task

Define a function to binarize age

In [ ]:
def categorize_age(age):  #shifting task from regression to classification; model will have to predict which of two age groups belongs to
    if age <= 11:
        return 0
    elif age >= 12:
        return 1

Binarizing age to obtain two classes

In [ ]:
df['Age_Category'] = df['Age'].apply(categorize_age)
#df['Age_Category'] = df['Age_Category'].astype(int)
# Count the occurrences of each unique value in the 'Age_Category' column
age_category_counts = df['Age_Category'].value_counts()

# Display the counts
print("Number of rows with Age_Category equal to 0:", age_category_counts[0])
print("Number of rows with Age_Category equal to 1:", age_category_counts[1])

Number of rows with Age_Category equal to 0: 32
Number of rows with Age_Category equal to 1: 40


A quick quality control

In [ ]:
df.tail(10)

Extracting "Age_category" and removing the "Age", "Sex" columns from the data

In [ ]:
y = df['Age_Category'] #new binary labels
X = df.drop(columns=['Age_Category','Age','Sex']) #matrix cleaned by irrelevant or confusing columns such as SEx and age

# Convert the 'Age_Category' column to integer type
#df['Age_Category'] = df['Age_Category'].astype(int)

print('The whole dataset contains ' + str(X.shape[0]) + ' subjects')
print('The age prediction will be performed using ' + str(X.shape[1]) + ' MRI-derived features')

The whole dataset contains 72 subjects
The age prediction will be performed using 33 MRI-derived features


A quick quality control

In [ ]:
X.tail(10)

In [ ]:
y.tail(10)

## Classification task

### Logistic regression using a CV scheme

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate, KFold
from sklearn.linear_model import LogisticRegression
# Setting the seed of the random generator
SEED = 42

# Setting the number of folds
n_folds = 5

# Creating the estimator: clf is the tool used for classification that uses an empty untrained logistic regression algorithm
clf = LogisticRegression()

# Creating the splitter: using K-Fold cross validation, with K-1 folds used as training set; shuffle allows to mix data preventing bias (if data were sorted based on age g.e.)
cv = KFold(n_splits=n_folds, shuffle=True, random_state=SEED)

In [ ]:
# Print the generated splits
for train_index, test_index in cv.split(X):
    print("Train:", train_index, " Test:", test_index)

In [ ]:
score = cross_validate(clf, X=X, y=y, cv=cv, return_train_score=True, return_estimator=True, scoring = 'roc_auc')
#cross validate actually implements the cross validation: return train score to make sure if model is learning too much from training data (overfitting)
#the scoring metric used is ROC AUC --> looks how confident model is in prediction, measuring tradeoff btw sensitivity and false positive across different thresholds
print("This is the score object:")
print (score)

print("Average AUC training set:", np.mean(score['train_score']))
print("Average AUC test set:", np.mean(score['test_score']))

This is the score object:
{'fit_time': array([0.0283823 , 0.01262784, 0.01180935, 0.00720763, 0.01095533]), 'score_time': array([0.00997519, 0.00458217, 0.0045197 , 0.00430274, 0.00488877]), 'estimator': [LogisticRegression(), LogisticRegression(), LogisticRegression(), LogisticRegression(), LogisticRegression()], 'test_score': array([0.90740741, 1.        , 1.        , 1.        , 0.91666667]), 'train_score': array([0.98014888, 0.96464646, 0.96848485, 0.96415771, 0.98317308])}
Average AUC training set: 0.9721221959044539
Average AUC test set: 0.9648148148148149


In [ ]:
print(type(score))
print(type(score['train_score']))

### Logistic Regression with hyperparameter C (Complexity) using a nested CV scheme

In [ ]:
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error
# Setting the seed of the random generator
SEED = 42

# Setting the number of folds of both outer and inner k-fold CV
outer_n_folds = 5
inner_n_folds = 5

# Setting the possible values of the C hyperparameter to control weight decay, high regularization for small coefficient 0.1
C = [0.1, 1, 10]

# Creating the splitters
outer_cv = KFold(n_splits=outer_n_folds, shuffle=True, random_state=SEED)
inner_cv = KFold(n_splits=inner_n_folds, shuffle=True, random_state=SEED)

# Creating the estimator
clf = LogisticRegression(max_iter=1000)

# Defining the grid of hyperparameter values
p_grid = [{'C': C}]

#gridsearch to find optimal value for hyperpar C; while nested score looks in outer loop to perform an unbiased evaluation, evaluating with data it has never seen before
clf_gs = GridSearchCV(clf, param_grid=p_grid, cv=inner_cv, refit='roc_auc', scoring='roc_auc', verbose = 4)
nested_score = cross_validate(clf_gs, X=X, y=y, cv=outer_cv, return_train_score=True, return_estimator=True, scoring = 'roc_auc')

#print(np.abs(nested_score['train_score']))
#print(np.abs(nested_score['test_score']))  --> the numbers I see there are AUC scores not MAE: IF HIGH MODEL ROBUST
print("Average MAE train:", np.mean(np.abs(nested_score['train_score'])), "years")
print("Average MAE test:", np.mean(np.abs(nested_score['test_score'])), "years")

In [ ]:
print(type(p_grid))
print(type(p_grid[0]))